In [ ]:
#---------------------------
#----MARCO ALARCON CRESPO---
#---------------------------

from abc import ABC, abstractmethod
from typing import Any


# ==========================================================
#                  PRODUCTOS ABSTRACTOS
# ==========================================================

class StorageService(ABC):

    @abstractmethod
    def upload_file(self, filename: str, data: bytes) -> str:
        pass


class NotificationService(ABC):

    @abstractmethod
    def send_alert(self, message: str) -> bool:
        pass


class AuthValidator(ABC):

    @abstractmethod
    def validate_token(self, token: str) -> bool:
        pass

In [ ]:
# ==========================================================
#                  FÁBRICA ABSTRACTA
# ==========================================================

class CloudInfrastructureFactory(ABC):

    @abstractmethod
    def create_storage(self) -> StorageService:
        pass

    @abstractmethod
    def create_notification(self) -> NotificationService:
        pass

    @abstractmethod
    def create_auth(self) -> AuthValidator:
        pass

In [ ]:
# ==========================================================
#     IMPLEMENTACIÓN DE PRODUCTOS Y FÁBRICAS CONCRETAS
# ==========================================================

# ---------------- AWS ----------------

class S3StorageService(StorageService):

    def upload_file(self, filename: str, data: bytes) -> str:
        print(f"Subiendo {filename} a AWS S3...")
        return f"Archivo {filename} almacenado en S3"


class SNSNotificationService(NotificationService):

    def send_alert(self, message: str) -> bool:
        print(f"AWS SNS: {message}")
        return True


class IAMAuthValidator(AuthValidator):

    def validate_token(self, token: str) -> bool:
        print("Validando token mediante AWS IAM...")
        return True


class AWSInfrastructureFactory(CloudInfrastructureFactory):

    def create_storage(self) -> StorageService:
        return S3StorageService()

    def create_notification(self) -> NotificationService:
        return SNSNotificationService()

    def create_auth(self) -> AuthValidator:
        return IAMAuthValidator()


# ---------------- Azure ----------------

class BlobStorageService(StorageService):

    def upload_file(self, filename: str, data: bytes) -> str:
        print(f"Subiendo {filename} a Azure Blob Storage...")
        return f"Archivo {filename} almacenado en Blob Storage"


class EventGridNotificationService(NotificationService):

    def send_alert(self, message: str) -> bool:
        print(f"Azure Event Grid: {message}")
        return True


class EntraAuthValidator(AuthValidator):

    def validate_token(self, token: str) -> bool:
        print("Validando token mediante Microsoft Entra...")
        return True


class AzureInfrastructureFactory(CloudInfrastructureFactory):

    def create_storage(self) -> StorageService:
        return BlobStorageService()

    def create_notification(self) -> NotificationService:
        return EventGridNotificationService()

    def create_auth(self) -> AuthValidator:
        return EntraAuthValidator()



In [ ]:
# ==========================================================
#                      CÓDIGO CLIENTE
# ==========================================================

class CloudDeploymentManager:

    def __init__(self, factory: CloudInfrastructureFactory) -> None:

        self.storage = factory.create_storage()
        self.notification = factory.create_notification()
        self.auth = factory.create_auth()

    def deploy_environment(self, config_data: str) -> None:

        token = config_data

        if self.auth.validate_token(token):

            print("Token válido.")

            self.storage.upload_file(
                "config.txt",
                config_data.encode()
            )

            self.notification.send_alert(
                "Entorno desplegado correctamente."
            )

        else:

            print("Token inválido.")


In [ ]:
#==========================================================
#                       PRUEBA AWS
# ==========================================================

aws_factory = AWSInfrastructureFactory()

aws_manager = CloudDeploymentManager(aws_factory)

aws_manager.deploy_environment("AWS-12345")

Validando token mediante AWS IAM...
Token válido.
Subiendo config.txt a AWS S3...
AWS SNS: Entorno desplegado correctamente.


In [ ]:
# ==========================================================
#                     PRUEBA AZURE
# ==========================================================

azure_factory = AzureInfrastructureFactory()

azure_manager = CloudDeploymentManager(azure_factory)

azure_manager.deploy_environment("AZURE-12345")

Validando token mediante Microsoft Entra...
Token válido.
Subiendo config.txt a Azure Blob Storage...
Azure Event Grid: Entorno desplegado correctamente.
